In [1]:
import numpy as np
import pandas as pd
from scapy.all import rdpcap, raw
import matplotlib.pyplot as plt
import seaborn as sns



In [13]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            31Gi       6.1Gi        23Gi       560Mi       2.8Gi        24Gi
Swap:           14Gi       2.1Gi        12Gi


cara PALING JELAS & PRAKTIS untuk melihat variabel apa saja yang ada di RAM dan seberapa besar ukurannya

In [12]:
%whos


Variable           Type        Data/Info
----------------------------------------
data               NpzFile     NpzFile '/home/dani/Docum<...>, y_train, x_test, y_test
gc                 module      <module 'gc' (built-in)>
keep               set         {''}
name               str         keep
np                 module      <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd                 module      <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
plt                module      <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
raw                function    <function raw at 0x7dd8579b9620>
rdpcap             function    <function rdpcap at 0x7dd855dfb380>
sns                module      <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>
train_test_split   function    <function train_test_split at 0x7dd89520b2e0>
val                set         {''}


Cara PALING AKURAT untuk NumPy array

In [ ]:
import numpy as np

for name, val in globals().items():
    if isinstance(val, np.ndarray):
        print(f"{name:30s} {val.nbytes/1024/1024:.2f} MB  shape={val.shape}")


Untuk np.memmap:

In [ ]:
for name, val in globals().items():
    if isinstance(val, np.memmap):
        print(f"{name:30s} memmap (disk-backed) shape={val.shape}")


HAPUS variabel dari RAM

In [ ]:
del Xtrain1, Xtrain2, Xtrain3
del Xval1, Xval2, Xval3
del Xtest1, Xtest2, Xtest3
import gc
gc.collect()

PAKSA LEPAS RAM (WAJIB setelah del)

In [ ]:
import gc
gc.collect()


HAPUS SEMUA VARIABEL BESAR OTOMATIS
Hapus semua NumPy array & memmap kecuali yang kamu mau simpan.

In [ ]:
import numpy as np, gc

keep = {"X_train_mm", "X_val_mm", "X_test_mm", 
        "y_trainbinary", "y_valbinary", "y_seq_test"}

for name in list(globals()):
    if name not in keep:
        val = globals()[name]
        if isinstance(val, (np.ndarray, np.memmap)):
            del globals()[name]

gc.collect()


234

In [2]:
# --- KONFIGURASI DAN PATH FILE ---
BASE_PATH = '/home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/'
TRAIN_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_19_50_training.pcap'
TRAIN_LABELS_PATH = BASE_PATH + 'y_train.csv'

# Kita definisikan juga path untuk data testing untuk nanti
TEST_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_20_04_test.pcap'
TEST_LABELS_PATH = BASE_PATH + 'y_test.csv'

print("Langkah 1 Selesai: Library dan path sudah siap.")

Langkah 1 Selesai: Library dan path sudah siap.


In [3]:
# --- MEMUAT LABEL TRAINING ---
print("Memuat file label dari:", TRAIN_LABELS_PATH)
df_labels_train = pd.read_csv(TRAIN_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train1 = df_labels_train.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train1)} label.")
print("Contoh 5 label pertama:", labels_train1[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 1].value_counts())

Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_train.csv

Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      954912
Abnormal    248825
Name: count, dtype: int64


In [4]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train2 = df_labels_train.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train2)} label.")
print("Contoh 5 label pertama:", labels_train2[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 2].value_counts())


Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    954912
C_D        85466
P_I        64635
F_I        35112
M_F        33765
C_R        29847
Name: count, dtype: int64


In [5]:

def extract_packets(pcap_path, n_bytes=64):
    packets = rdpcap(pcap_path)
    data = []

    for pkt in packets:
        pkt_bytes = raw(pkt)

        if len(pkt_bytes) < n_bytes:
            pkt_bytes = pkt_bytes + bytes(n_bytes - len(pkt_bytes))  # zero padding
        else:
            pkt_bytes = pkt_bytes[:n_bytes]  # truncate

        data.append(np.frombuffer(pkt_bytes, dtype=np.uint8))

    return np.array(data)  # (num_packets, 64)


In [6]:
def build_sequences_multiclass(X_pkt, y_pkt, window=64, step=1):
    X_seq, y_seq = [], []

    for i in range(0, len(X_pkt) - window + 1, step):
        window_data = X_pkt[i:i+window]
        window_label = y_pkt[i:i+window]

        X_seq.append(window_data.T)

        # buang normal kalau ada attack
        attack_only = window_label[window_label > 0]

        if len(attack_only) > 0:
            vals, cnts = np.unique(attack_only, return_counts=True)
            y_seq.append(vals[np.argmax(cnts)])
        else:
            y_seq.append(0)

    return np.array(X_seq), np.array(y_seq)


training binary class

In [7]:
x_pkt_train=extract_packets(TRAIN_PCAP_PATH)

In [8]:
multi_labels_train = df_labels_train.iloc[:, 2]
y_label_train  = multi_labels_train.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values


In [9]:

x_seq_train, y_seq_train = build_sequences_multiclass(x_pkt_train,y_label_train)

In [10]:
x_seq_train.shape, y_seq_train.shape

((1203674, 64, 64), (1203674,))

training multiclass

In [11]:
np.unique(y_seq_train)

array([0, 1, 2, 3, 4, 5])

test binary class

In [12]:
# --- MEMUAT LABEL Testing ---
print("Memuat file label dari:", TEST_LABELS_PATH)
df_labels_test = pd.read_csv(TEST_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test1 = df_labels_test.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test1)} label.")
print("Contoh 5 label pertama:", labels_test1[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 1].value_counts())
print(df_labels_test.head())


Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_test.csv

Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      660777
Abnormal    130834
Name: count, dtype: int64
   0       1       2
0  1  Normal  Normal
1  2  Normal  Normal
2  3  Normal  Normal
3  4  Normal  Normal
4  5  Normal  Normal


In [13]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test2 = df_labels_test.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test2)} label.")
print("Contoh 5 label pertama:", labels_test2[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 2].value_counts())


Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    660777
C_D        41203
C_R        29847
P_I        26013
F_I        16962
M_F        16809
Name: count, dtype: int64


In [14]:
x_pkt_test=extract_packets(TEST_PCAP_PATH)

In [15]:
multi_labels_test = df_labels_test.iloc[:, 2]
y_label_test  = multi_labels_test.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values


In [16]:
y_label_testmulti = multi_labels_test.map({'Normal': 0, 'C_D': 1, 'P_I': 2, 'F_I': 3, 'M_F': 4, 'C_R': 5}).values
np.unique(y_label_testmulti)

label_map = {
    0: 'Normal',
    1: 'C_D',
    2: 'P_I',
    3: 'F_I',
    4: 'M_F',
    5: 'C_R'
}

labels, counts = np.unique(y_label_testmulti, return_counts=True)

for l, c in zip(labels, counts):
    print(f"{label_map[l]} ({l}): {c}")


Normal (0): 660777
C_D (1): 41203
P_I (2): 26013
F_I (3): 16962
M_F (4): 16809
C_R (5): 29847


In [17]:

x_seq_test, y_seq_test = build_sequences_multiclass(x_pkt_test, y_label_testmulti)
x_seq_test.shape, y_seq_test.shape

((791548, 64, 64), (791548,))

In [18]:
np.unique(y_seq_test)

array([0, 1, 2, 3, 4, 5])

multiclass

In [4]:
np.savez_compressed(
    "preprocessingW64S1multifloat32/tow_ids_multiclass_preprocessedW64S1v1.npz",
    x_train=x_seq_train,
    y_train=y_seq_train,
    x_test=x_seq_test,
    y_test=y_seq_test
)


NameError: name 'x_seq_train' is not defined

In [5]:
data = np.load("/home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/preprocessingW64S1multifloat32/tow_ids_multiclass_preprocessedW64S1v1.npz")

x_seq_train = data["x_train"]
y_seq_train = data["y_train"]
x_seq_test = data["x_test"]
y_seq_test = data["y_test"]
    
print(x_seq_train.shape, y_seq_train.shape)
print(x_seq_test.shape, y_seq_test.shape)


(1203674, 64, 64) (1203674,)
(791548, 64, 64) (791548,)


split

In [6]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    x_seq_train, y_seq_train,
    test_size=0.3,
    stratify=y_seq_train,
    random_state=42
)

In [7]:
print("Original:")
print(x_seq_train.shape, y_seq_train.shape)

print("\nAfter split:")
print("Train:", x_train.shape, y_train.shape)
print("Val  :", x_val.shape, y_val.shape)


Original:
(1203674, 64, 64) (1203674,)

After split:
Train: (842571, 64, 64) (842571,)
Val  : (361103, 64, 64) (361103,)


In [ ]:
import numpy as np

label_map = {
    0: 'Normal',
    1: 'C_D',
    2: 'P_I',
    3: 'F_I',
    4: 'M_F',
    5: 'C_R'
}

def print_label_dist(y, title):
    labels, counts = np.unique(y, return_counts=True)
    total = len(y)

    print(f"\n{title}")
    for l, c in zip(labels, counts):
        print(f"{label_map[l]} ({l}): {c} ({c/total:.2%})")

print_label_dist(y_seq_train, "Original")
print_label_dist(y_train, "Train")
print_label_dist(y_val, "Validation")



Original
Normal (0): 550090 (45.70%)
C_D (1): 178730 (14.85%)
P_I (2): 193460 (16.07%)
F_I (3): 75316 (6.26%)
M_F (4): 104429 (8.68%)
C_R (5): 101649 (8.44%)

Train
Normal (0): 385063 (45.70%)
C_D (1): 125111 (14.85%)
P_I (2): 135422 (16.07%)
F_I (3): 52721 (6.26%)
M_F (4): 73100 (8.68%)
C_R (5): 71154 (8.44%)

Validation
Normal (0): 165027 (45.70%)
C_D (1): 53619 (14.85%)
P_I (2): 58038 (16.07%)
F_I (3): 22595 (6.26%)
M_F (4): 31329 (8.68%)
C_R (5): 30495 (8.44%)


normalize

In [24]:
import numpy as np

def split_X_into_n_parts(X, n_parts=3):
    total = X.shape[0]
    indices = np.array_split(np.arange(total), n_parts)
    return [X[idx] for idx in indices]


In [ ]:
x_train_parts = split_X_into_n_parts(x_train, n_parts=3)

for i, Xp in enumerate(x_train_parts):
    print(f"Train X part {i+1}: {Xp.shape}")


Train X part 1: (280857, 64, 64)
Train X part 2: (280857, 64, 64)
Train X part 3: (280857, 64, 64)


In [26]:
x_test_parts = split_X_into_n_parts(x_seq_test, n_parts=3)

for i, Xp in enumerate(x_test_parts):
    print(f"Test X part {i+1}: {Xp.shape}")


Test X part 1: (263850, 64, 64)
Test X part 2: (263849, 64, 64)
Test X part 3: (263849, 64, 64)


In [ ]:
x_val_parts = split_X_into_n_parts(x_val, n_parts=3)

for i, Xp in enumerate(x_val_parts):
    print(f"Val X part {i+1}: {Xp.shape}")


Val X part 1: (120368, 64, 64)
Val X part 2: (120368, 64, 64)
Val X part 3: (120367, 64, 64)


In [28]:
import numpy as np
import gc
def normalize_minmax(X):
    return X.astype(np.float32) / 255.0



In [29]:
print("Loading train part 1...")
Xp = x_train_parts[0]        # hanya 1 part di RAM

print("Normalizing train part 1...")
Xp_norm = normalize_minmax(Xp)

print("Saving train part 1...")
np.savez_compressed("preprocessingW64S1multifloat32/x_train_norm_part_1multi.npz", x=Xp_norm)

# Bersihkan RAM
del Xp, Xp_norm
gc.collect()

print("DONE train part 1")


Loading train part 1...
Normalizing train part 1...
Saving train part 1...
DONE train part 1


In [30]:
print("Loading train part 2...")
Xp = x_train_parts[1]

print("Normalizing train part 2...")
Xp_norm = normalize_minmax(Xp)

print("Saving train part 2...")
np.savez_compressed("preprocessingW64S1multifloat32/x_train_norm_part_2multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE train part 2")


Loading train part 2...
Normalizing train part 2...
Saving train part 2...
DONE train part 2


In [31]:
print("Loading train part 3...")
Xp = x_train_parts[2]

print("Normalizing train part 3...")
Xp_norm = normalize_minmax(Xp)

print("Saving train part 3...")
np.savez_compressed("preprocessingW64S1multifloat32/x_train_norm_part_3multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE train part 3")


Loading train part 3...
Normalizing train part 3...
Saving train part 3...
DONE train part 3


In [32]:
print("Loading val part 1...")
Xp = x_val_parts[0]        # hanya 1 part di RAM

print("Normalizing val part 1...")
Xp_norm = normalize_minmax(Xp)

print("Saving val part 1...")
np.savez_compressed("preprocessingW64S1multifloat32/x_val_norm_part_1multi.npz", x=Xp_norm)

# Bersihkan RAM
del Xp, Xp_norm
gc.collect()

print("DONE val part 1")

Loading val part 1...
Normalizing val part 1...
Saving val part 1...
DONE val part 1


In [33]:
print("Loading val part 2...")
Xp = x_val_parts[1]

print("Normalizing val part 2...")
Xp_norm = normalize_minmax(Xp)

print("Saving val part 2...")
np.savez_compressed("preprocessingW64S1multifloat32/x_val_norm_part_2multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE val part 2")


Loading val part 2...
Normalizing val part 2...
Saving val part 2...
DONE val part 2


In [34]:
print("Loading val part 3...")
Xp = x_val_parts[2]

print("Normalizing val part 3...")
Xp_norm = normalize_minmax(Xp)

print("Saving val part 3...")
np.savez_compressed("preprocessingW64S1multifloat32/x_val_norm_part_3multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE val part 3")


Loading val part 3...
Normalizing val part 3...
Saving val part 3...
DONE val part 3


In [35]:
print("Loading test part 1...")
Xp = x_test_parts[0]

print("Normalizing test part 1...")
Xp_norm = normalize_minmax(Xp)

print("Saving test part 1...")
np.savez_compressed("preprocessingW64S1multifloat32/x_test_norm_part_1multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE test part 1")


Loading test part 1...
Normalizing test part 1...
Saving test part 1...
DONE test part 1


In [36]:
print("Loading test part 2...")
Xp = x_test_parts[1]

print("Normalizing test part 2...")
Xp_norm = normalize_minmax(Xp)

print("Saving test part 2...")
np.savez_compressed("preprocessingW64S1multifloat32/x_test_norm_part_2multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE test part 2")


Loading test part 2...
Normalizing test part 2...
Saving test part 2...
DONE test part 2


In [37]:
print("Loading test part 3...")
Xp = x_test_parts[2]

print("Normalizing test part 3...")
Xp_norm = normalize_minmax(Xp)

print("Saving test part 3...")
np.savez_compressed("preprocessingW64S1multifloat32/x_test_norm_part_3multi.npz", x=Xp_norm)

del Xp, Xp_norm
gc.collect()

print("DONE test part 3")


Loading test part 3...
Normalizing test part 3...
Saving test part 3...
DONE test part 3


In [38]:
datatrainpart1 = np.load("preprocessingW64S1multifloat32/x_train_norm_part_1multi.npz")
Xtrain = datatrainpart1["x"]
datavalpart1 = np.load("preprocessingW64S1multifloat32/x_val_norm_part_1multi.npz")
Xval = datavalpart1["x"]
datatestpart1 = np.load("preprocessingW64S1multifloat32/x_test_norm_part_1multi.npz")
Xtest = datatestpart1["x"]

In [39]:
print("Shape:", Xtrain.shape)
print("Dtype:", Xtrain.dtype)
print("Shape:", Xval.shape)
print("Dtype:", Xval.dtype)
print("Shape:", Xtest.shape)
print("Dtype:", Xtest.dtype)


Shape: (280857, 64, 64)
Dtype: float32
Shape: (120368, 64, 64)
Dtype: float32
Shape: (263850, 64, 64)
Dtype: float32


In [44]:
print("Min:", Xtrain.min())
print("Max:", Xtrain.max())
print("Min:", Xval.min())
print("Max:", Xval.max())
print("Min:", Xtest.min())
print("Max:", Xtest.max())


Min: 0.0
Max: 1.0
Min: 0.0
Max: 1.0
Min: 0.0
Max: 1.0


In [41]:
sample = Xtrain[:1000]          # atau random slice
uniq_vals = np.unique(sample)

print("Unique (sample):", uniq_vals[:10])
print("Total unique (sample):", len(uniq_vals))


Unique (sample): [0.         0.00392157 0.00784314 0.01176471 0.01568628 0.01960784
 0.02352941 0.02745098 0.03137255 0.03529412]
Total unique (sample): 256


In [42]:
sample = Xval[:1000]          # atau random slice
uniq_vals = np.unique(sample)

print("Unique (sample):", uniq_vals[:10])
print("Total unique (sample):", len(uniq_vals))

Unique (sample): [0.         0.00392157 0.00784314 0.01176471 0.01568628 0.01960784
 0.02352941 0.02745098 0.03137255 0.03529412]
Total unique (sample): 256


In [43]:
sample = Xtest[:1000]          # atau random slice
uniq_vals = np.unique(sample)

print("Unique (sample):", uniq_vals[:10])
print("Total unique (sample):", len(uniq_vals))

Unique (sample): [0.         0.00392157 0.00784314 0.01176471 0.01568628 0.01960784
 0.02352941 0.02745098 0.03137255 0.03529412]
Total unique (sample): 256


In [11]:
datatrainpart1 = np.load("preprocessingW64S1multifloat32/x_train_norm_part_1multi.npz")
Xtrain1 = datatrainpart1["x"]
datatrainpart2 = np.load("preprocessingW64S1multifloat32/x_train_norm_part_2multi.npz")
Xtrain2 = datatrainpart2["x"]
datatrainpart3 = np.load("preprocessingW64S1multifloat32/x_train_norm_part_3multi.npz")
Xtrain3 = datatrainpart3["x"]
datavalpart1 = np.load("preprocessingW64S1multifloat32/x_val_norm_part_1multi.npz")
Xval1 = datavalpart1["x"]
datavalpart2 = np.load("preprocessingW64S1multifloat32/x_val_norm_part_2multi.npz")
Xval2 = datavalpart2["x"]
datavalpart3 = np.load("preprocessingW64S1multifloat32/x_val_norm_part_3multi.npz")
Xval3 = datavalpart3["x"]
datatestpart1 = np.load("preprocessingW64S1multifloat32/x_test_norm_part_1multi.npz")
Xtest1 = datatestpart1["x"]
datatestpart2 = np.load("preprocessingW64S1multifloat32/x_test_norm_part_2multi.npz")
Xtest2 = datatestpart2["x"]
datatestpart3 = np.load("preprocessingW64S1multifloat32/x_test_norm_part_3multi.npz")
Xtest3 = datatestpart3["x"]


gabung

In [12]:
train_total = Xtrain1.shape[0] + Xtrain2.shape[0] + Xtrain3.shape[0]
val_total   = Xval1.shape[0]   + Xval2.shape[0]   + Xval3.shape[0]
test_total  = Xtest1.shape[0]  + Xtest2.shape[0]  + Xtest3.shape[0]

print("Train total:", train_total)
print("Val total  :", val_total)
print("Test total :", test_total)


Train total: 842571
Val total  : 361103
Test total : 791548


In [16]:
import numpy as np

X_train_mm = np.memmap(
    "preprocessingW64S1multifloat32/X_train_norm_float32.memmap",
    dtype=np.float32,
    mode="w+",
    shape=(train_total, 64, 64)
)

X_val_mm = np.memmap(
    "preprocessingW64S1multifloat32/X_val_norm_float32.memmap",
    dtype=np.float32,
    mode="w+",
    shape=(val_total, 64, 64)
)

X_test_mm = np.memmap(
    "preprocessingW64S1multifloat32/X_test_norm_float32.memmap",
    dtype=np.float32,
    mode="w+",
    shape=(test_total, 64, 64)
)


In [17]:
start = 0
for Xp in [Xtrain1, Xtrain2, Xtrain3]:
    end = start + Xp.shape[0]
    X_train_mm[start:end] = Xp
    start = end


In [18]:
start = 0
for Xp in [Xval1, Xval2, Xval3]:
    end = start + Xp.shape[0]
    X_val_mm[start:end] = Xp
    start = end


In [19]:
start = 0
for Xp in [Xtest1, Xtest2, Xtest3]:
    end = start + Xp.shape[0]
    X_test_mm[start:end] = Xp
    start = end


In [8]:
np.savez_compressed(
    "preprocessingW64S1multifloat32/labels_multi_split.npz",
    y_train=y_train.astype(np.int64),
    y_val=y_val.astype(np.int64),
    y_test=y_seq_test.astype(np.int64)
)
